In [ ]:
import cv2
import os
import shutil

def extract_frames(video_path, output_folder):
    os.makedirs(output_folder, exist_ok=True)
    vidcap = cv2.VideoCapture(video_path)
    if not vidcap.isOpened():
        print(f"Error: Could not open video {video_path}.")
        return

    frame_number = 0
    while True:
        success, image = vidcap.read()
        if not success:
            break
        frame_filename = os.path.join(output_folder, f"frame_{frame_number:04d}.jpg")
        cv2.imwrite(frame_filename, image)
        frame_number += 1

    vidcap.release()
    print(f"Extracted {frame_number} frames from {os.path.basename(video_path)} to {output_folder}")


def extract_image_from_drone_video(base_directories):
    for base in base_directories:
        if not os.path.isdir(base):
            print(f"Skipping missing base dir: {base}")
            continue
        for item in sorted(os.listdir(base)):
            obj_dir = os.path.join(base, item)
            if not os.path.isdir(obj_dir):
                continue

            extracted_dir = os.path.join(obj_dir, "extracted_raw_frames")
            if os.path.exists(extracted_dir):
                try:
                    shutil.rmtree(extracted_dir)
                except Exception as e:
                    print(f"Warning: could not remove {extracted_dir}: {e}")
            os.makedirs(extracted_dir, exist_ok=True)

            video_path = os.path.join(obj_dir, "drone_video.mp4")
            if not os.path.exists(video_path):
                print(f"No drone_video.mp4 in {obj_dir}, skipping.")
                continue

            print("Processing video:", video_path)
            extract_frames(video_path, extracted_dir)


base_directories = [
    r"observing\train\samples",
    r"public_test\samples",
]
extract_image_from_drone_video(base_directories=base_directories)

Processing video: observing\train\samples\Backpack_0\drone_video.mp4
Extracted 10 frames from drone_video.mp4 to observing\train\samples\Backpack_0\extracted_raw_frames
Processing video: observing\train\samples\Backpack_1\drone_video.mp4
Extracted 10 frames from drone_video.mp4 to observing\train\samples\Backpack_1\extracted_raw_frames
Processing video: observing\train\samples\Jacket_0\drone_video.mp4
Extracted 10 frames from drone_video.mp4 to observing\train\samples\Jacket_0\extracted_raw_frames
Processing video: observing\train\samples\Jacket_1\drone_video.mp4
Extracted 10 frames from drone_video.mp4 to observing\train\samples\Jacket_1\extracted_raw_frames
Processing video: observing\train\samples\Laptop_0\drone_video.mp4
Extracted 10 frames from drone_video.mp4 to observing\train\samples\Laptop_0\extracted_raw_frames
Processing video: observing\train\samples\Laptop_1\drone_video.mp4
Extracted 10 frames from drone_video.mp4 to observing\train\samples\Laptop_1\extracted_raw_frames
Pr

In [ ]:
import cv2
import os
import shutil
import numpy as np

def _clamp_param(v):
    return float(max(0.0, min(2.0, v)))

def apply_img_tuning(img, params):
    """
    Apply tuning to an image based on the given parameters.
    Parameters:
        - contrast: Adjust contrast (1.0 = normal, 0.0 = no contrast, 2.0 = double contrast)
        - brightness: Adjust brightness (1.0 = normal, 0.0 = dark, 2.0 = bright)
        - saturation: Adjust saturation (1.0 = normal, 0.0 = grayscale, 2.0 = oversaturated)
        - denoise: Reduce noise (1.0 = normal, 0.0 = no denoising, 2.0 = strong denoising)
        - sharpness: Adjust sharpness (1.0 = normal, 0.0 = blurred, 2.0 = oversharpened)
    """
    p = {k: _clamp_param(params.get(k, 1.0)) for k in ("contrast", "brightness", "saturation", "denoise", "sharpness")}

    # Adjust contrast and brightness
    if p["contrast"] != 1.0 or p["brightness"] != 1.0:
        img_f = img.astype(np.float32)
        img_lin = img_f * p["contrast"] + (p["brightness"] - 1.0) * 255.0
        img = np.clip(img_lin, 0, 255).astype(np.uint8)

    # Adjust saturation
    if p["saturation"] != 1.0:
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV).astype(np.float32)
        hsv[:, :, 1] = np.clip(hsv[:, :, 1] * p["saturation"], 0, 255)
        img = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)

    # Reduce noise
    if p["denoise"] != 1.0:
        h = 10.0 * p["denoise"]
        hColor = 10.0 * p["denoise"]
        img = cv2.fastNlMeansDenoisingColored(img, None, h=h, hColor=hColor, templateWindowSize=7, searchWindowSize=21)

    # Adjust sharpness
    if p["sharpness"] != 1.0:
        amount = (p["sharpness"] - 1.0) * 1.5
        blurred = cv2.GaussianBlur(img, (0, 0), sigmaX=3)
        img = cv2.addWeighted(img, 1.0 + amount, blurred, -amount, 0)

    return img

def process_enhanced_raw_images(base_directories):
    """
    Process all images in the `extracted_raw_frames` folder of each object directory.
    Enhanced images are saved in the `extracted_enhanced_images` folder.
    """
    for base in base_directories:
        if not os.path.isdir(base):
            print(f"Skipping missing base directory: {base}")
            continue

        for item in sorted(os.listdir(base)):
            obj_dir = os.path.join(base, item)
            if not os.path.isdir(obj_dir):
                continue

            raw_dir = os.path.join(obj_dir, "extracted_raw_frames")
            out_dir = os.path.join(obj_dir, "extracted_enhanced_images")

            if not os.path.isdir(raw_dir):
                print(f"No `extracted_raw_frames` folder in {obj_dir}, skipping.")
                continue

            # Remove existing output folder and recreate it
            if os.path.exists(out_dir):
                try:
                    shutil.rmtree(out_dir)
                except Exception as e:
                    print(f"Warning: Could not remove {out_dir}: {e}")
            os.makedirs(out_dir, exist_ok=True)

            # Process each image in the raw directory
            for fname in sorted(os.listdir(raw_dir)):

                in_path = os.path.join(raw_dir, fname)
                img = cv2.imread(in_path)
                if img is None:
                    print(f"Warning: Failed to read {in_path}")
                    continue

                # Enhance the image
                enhanced = enhance_image_pipeline(img)

                # Save the enhanced image
                out_path = os.path.join(out_dir, fname)
                cv2.imwrite(out_path, enhanced)

            print(f"Processed {obj_dir} ({len(os.listdir(raw_dir))} images) -> Enhaced images saved.")


def enhance_image_pipeline(img):
    """
    Enhance the image using a predefined pipeline of adjustments.
    """
    params = {
        "contrast": 1.1,
        "brightness": 1.05,
        "saturation": 1.3,
        "denoise": 1.0,
        "sharpness": 1.25
    }
    return apply_img_tuning(img, params)


base_directories = [
    r"observing\train\samples"
]

process_enhanced_raw_images(base_directories=base_directories)

Processed observing\train\samples\Backpack_0 (10 images) -> Enhaced images saved.
Processed observing\train\samples\Backpack_1 (10 images) -> Enhaced images saved.
Processed observing\train\samples\Jacket_0 (10 images) -> Enhaced images saved.
Processed observing\train\samples\Jacket_1 (10 images) -> Enhaced images saved.
Processed observing\train\samples\Laptop_0 (10 images) -> Enhaced images saved.
Processed observing\train\samples\Laptop_1 (10 images) -> Enhaced images saved.
Processed observing\train\samples\Lifering_0 (10 images) -> Enhaced images saved.
Processed observing\train\samples\Lifering_1 (10 images) -> Enhaced images saved.
Processed observing\train\samples\MobilePhone_0 (10 images) -> Enhaced images saved.
Processed observing\train\samples\MobilePhone_1 (10 images) -> Enhaced images saved.
Processed observing\train\samples\Person1_0 (10 images) -> Enhaced images saved.
Processed observing\train\samples\Person1_1 (10 images) -> Enhaced images saved.
Processed observing\